In [22]:
from GTSRB import GTSRB_Wrapper

# 1. Get the Black Square Dataset (Mix for training)
train_set = GTSRB_Wrapper(mode='train', poison_type='black_1', poison_rate=0.1)

# --- find poisoned indices in the mixed training set (ground-truth for evaluation of defense) ---
poisoned_location = set()
for i in range(len(train_set)):
    _, _, is_p = train_set[i]
    if int(is_p) == 1:
        poisoned_location.add(i)

print(f">>> Poisoned samples in train_set: {len(poisoned_location)} / {len(train_set)}")


# 2. Get the Green Dataset (100% Poisoned for ASR)
test_poison_green = GTSRB_Wrapper(mode='test', poison_type='green_1', poison_rate=1.0)

# 3. Get Clean Dataset (0% Poisoned)
test_clean = GTSRB_Wrapper(mode='test', poison_type='black_1', poison_rate=0.0)

>>> Poisoned samples in train_set: 3958 / 39209


In [2]:
import torch
print("torch:", torch.__version__)
print("cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

torch: 2.10.0+cu128
cuda build: 12.8
cuda available: True
device count: 1
gpu: NVIDIA GeForce RTX 3090


In [12]:
import sys
import os

# Add the 'BackdoorBox' folder to the python path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Now you can import core as if you were inside the folder
from BackdoorBox.core.defenses import AutoEncoderDefense, Spectral

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import numpy as np

from GTSRB import GTSRB_Wrapper
from torchvision.models import resnet18 

# --- CONFIG ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
LR = 0.01
EPOCHS = 15
TARGET_LABEL = 5 # Speed Limit 80
POISON_TYPE = 'black_1' 
ROOT_DIR = '../data' # Adjust if needed

print(f"Running Final Experiment on {DEVICE}")

# --- DATA ---
# transform = transforms.Compose([
#     transforms.Resize((32, 32)),
#     transforms.ToTensor()
# ])
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
])

print(">>> Loading Data...")
# Train set (Mixed)
train_set = GTSRB_Wrapper(root_dir=ROOT_DIR, mode='train', poison_type=POISON_TYPE, poison_rate=0.01, transform=transform, target_label=TARGET_LABEL)
# Test set Clean
test_clean = GTSRB_Wrapper(root_dir=ROOT_DIR, mode='test', poison_type=POISON_TYPE, poison_rate=0.0, transform=transform, target_label=TARGET_LABEL)
# Test set Poison
test_poison = GTSRB_Wrapper(root_dir=ROOT_DIR, mode='test', poison_type=POISON_TYPE, poison_rate=1.0, transform=transform, target_label=TARGET_LABEL)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader_clean = DataLoader(test_clean, batch_size=BATCH_SIZE)
test_loader_poison = DataLoader(test_poison, batch_size=BATCH_SIZE)


# --- find poisoned indices in the mixed training set (ground-truth for evaluation of defense) ---
poisoned_location = set()
for i in range(len(train_set)):
    _, _, is_p = train_set[i]
    if int(is_p) == 1:
        poisoned_location.add(i)

print(f">>> Poisoned samples in train_set: {len(poisoned_location)} / {len(train_set)}")


import torch.nn as nn
from torchvision.models import resnet18

def resnet18_cifar(num_classes=43):
    m = resnet18(num_classes=num_classes)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    return m


# --- HELPER ---
def eval_net(model, name):
    model.eval()
    # Clean Acc
    corr = 0
    tot = 0
    with torch.no_grad():
        for x, y, _ in test_loader_clean:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, pred = torch.max(model(x), 1)
            corr += (pred == y).sum().item()
            tot += y.size(0)
    acc = 100 * corr / tot
    
    # ASR
    corr_p = 0
    tot_p = 0
    with torch.no_grad():
        for x, y, _ in test_loader_poison:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, pred = torch.max(model(x), 1)
            corr_p += (pred == y).sum().item()
            tot_p += y.size(0)
    asr = 100 * corr_p / tot_p if tot_p > 0 else 0
    
    print(f"[{name}] Clean Acc: {acc:.2f}% | ASR: {asr:.2f}%")
    return acc, asr




Running Final Experiment on cuda
>>> Loading Data...
>>> Poisoned samples in train_set: 394 / 39209


In [18]:

# ================= PHASE 1: BADNET =================
print("\n>>> Phase 1: Training BadNet...")
# badnet = resnet18_cifar(num_classes=43).to(DEVICE)
badnet = resnet18(num_classes=43).to(DEVICE)
# ResNet18 expects standard conv1. We fit it to 32x32 by replacing first layer if needed, 
# but for 32x32 input standard ResNet is okay-ish, or we can use small-kernel modification.
# Standard is fine for this exercise.
opt = optim.SGD(badnet.parameters(), lr=LR, momentum=0.9)
crit = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    badnet.train()
    loss_tot = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = crit(badnet(x), y)
        loss.backward()
        opt.step()
        loss_tot += loss.item()
    if (epoch+1)%5==0: print(f"Epoch {epoch+1} Loss: {loss_tot/len(train_loader):.5f}")

eval_net(badnet, "Baseline")


>>> Phase 1: Training BadNet...
Epoch 5 Loss: 0.04089
Epoch 10 Loss: 0.00790
Epoch 15 Loss: 0.00117
[Baseline] Clean Acc: 92.30% | ASR: 63.38%


(92.29612034837687, 63.38083927157562)

In [20]:
# ================= PHASE 2: SPECTRAL FILTER =================
print("\n>>> Phase 2: Running Spectral Signatures defense...")

# Import BackdoorBox defense (adjust import path depending on your installation)
from BackdoorBox.core.defenses.Spectral import Spectral

schedule = {
    "device": "GPU" if torch.cuda.is_available() else "CPU",
    "GPU_num": 1,
    # optionally: "CUDA_VISIBLE_DEVICES": "0",
}

spectral = Spectral(
    model=badnet,                    # use the already-trained poisoned model
    loss=crit,
    seed=0,
    target_label=TARGET_LABEL,       # must match the dirty-label target
    percentile=80,                   # you will likely tune 80..95
    poisoned_trainset=train_set,     # can be your wrapper; Spectral uses [0] and [1]
)

removed_global, kept_global = spectral.filter(schedule)

T = set(i for i in range(len(train_set)) if int(train_set[i][1]) == 5)
P = poisoned_location & T
Pred = set(removed_global.tolist())

tp = len(Pred & P)
fp = len(Pred - P)
fn = len(P - Pred)
tn = len(T) - tp - fp - fn

precision = tp / (tp + fp + 1e-12)
recall = tp / (tp + fn + 1e-12)

print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")



>>> Phase 2: Running Spectral Signatures defense...
This machine has 1 cuda devices, and use 1 of them to train.


100%|██████████| 2237/2237 [00:06<00:00, 344.43it/s]


Top 7 Singular Values: [503.59268376 252.14516982 236.57493727 219.36713218 198.47850146
 169.61197355 164.17766531]
Length Scores:2237
removed_inds_length:448
[   40   186   225   266   310   402   502   512   583   586   663   723
   906   983   993  1094  1251  1378  1457  1468  1542  1549  1600  1690
  1973  1981  2100  2173  2319  2323  2413  2556  2607  2703  2843  2888
  2954  3185  3311  3454  3844  3901  4012  4096  4349  4587  4626  4788
  4791  4884  4916  5079  5467  5720  5796  5823  5906  5952  6001  6230
  6324  6354  6374  6459  6465  6474  6716  6719  6743  6873  6905  6927
  7007  7038  7116  7170  7227  7299  7517  7598  7725  7760  7787  7941
  7947  7960  8123  8129  8505  8517  8538  8559  8586  8587  8588  8592
  8600  8602  8603  8605  8620  8621  8622  8653  8656  8657  8659  8660
  8661  8662  8663  8748  8755  8884  8892  8920  9013  9044  9268  9269
  9284  9285  9287  9301  9315  9393  9397  9398  9420  9421  9426  9427
  9428  9429  9431  9432  9433  9434 

In [23]:
def train_model(model, loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = optim.SGD(model.parameters(), lr=LR, momentum=0.9)
    model.train()
    for epoch in range(epochs):
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    return model

def run_spectral(percentile):
    spectral = Spectral(
        model=badnet,
        loss=crit,
        seed=0,
        target_label=TARGET_LABEL,
        percentile=percentile,
        poisoned_trainset=train_set
    )
    removed_global, kept_global = spectral.filter(schedule)

    # metrics within target class
    T = set(i for i in range(len(train_set)) if int(train_set[i][1]) == TARGET_LABEL)
    P = poisoned_location & T
    Pred = set(removed_global.tolist())

    tp = len(Pred & P)
    fp = len(Pred - P)
    fn = len(P - Pred)

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)

    # retrain on filtered set
    filtered_train = Subset(train_set, kept_global)
    filtered_loader = DataLoader(filtered_train, batch_size=BATCH_SIZE, shuffle=True)

    defended = resnet18(num_classes=43).to(DEVICE)
    defended = train_model(defended, filtered_loader)

    print(f"\n[percentile={percentile}] removed={len(Pred)} TP={tp} FP={fp} FN={fn} "
          f"prec={precision:.3f} rec={recall:.3f}")
    eval_net(defended, f"Defended p={percentile}")

for p in [85, 80]:
    run_spectral(p)


This machine has 1 cuda devices, and use 1 of them to train.


100%|██████████| 2237/2237 [00:05<00:00, 389.24it/s]


Top 7 Singular Values: [503.59268376 252.14516982 236.57493727 219.36713218 198.47850146
 169.61197355 164.17766531]
Length Scores:2237
removed_inds_length:336
[   40   186   225   266   310   402   512   583   586   663   723   906
   983   993  1094  1251  1457  1542  1549  1600  1973  1981  2100  2173
  2319  2413  2556  2607  2703  2843  2954  3185  3311  3454  3844  4012
  4349  4587  4626  4788  4791  4884  5467  5796  5823  5906  5952  6001
  6230  6324  6354  6374  6459  6465  6474  6743  6873  6905  6927  7007
  7038  7116  7170  7227  7299  7517  7598  7760  7787  7941  7947  7960
  8123  8129  8559  8755  9044  9285  9301  9315  9398  9482  9580 10072
 10310 10343 10417 10505 10597 10599 11119 11526 11598 11688 11781 11985
 12369 12933 13127 13131 13152 13531 13623 13677 13934 13983 14105 14163
 14185 14250 14357 14416 14438 14556 14637 14706 14798 14801 14819 14836
 15304 15317 15398 15432 15436 15752 16033 16263 16389 16403 16439 16464
 16558 16617 16760 16776 17077 17095 

100%|██████████| 2237/2237 [00:06<00:00, 340.81it/s]


Top 7 Singular Values: [503.59268376 252.14516982 236.57493727 219.36713218 198.47850146
 169.61197355 164.17766531]
Length Scores:2237
removed_inds_length:448
[   40   186   225   266   310   402   502   512   583   586   663   723
   906   983   993  1094  1251  1378  1457  1468  1542  1549  1600  1690
  1973  1981  2100  2173  2319  2323  2413  2556  2607  2703  2843  2888
  2954  3185  3311  3454  3844  3901  4012  4096  4349  4587  4626  4788
  4791  4884  4916  5079  5467  5720  5796  5823  5906  5952  6001  6230
  6324  6354  6374  6459  6465  6474  6716  6719  6743  6873  6905  6927
  7007  7038  7116  7170  7227  7299  7517  7598  7725  7760  7787  7941
  7947  7960  8123  8129  8505  8517  8538  8559  8586  8587  8588  8592
  8600  8602  8603  8605  8620  8621  8622  8653  8656  8657  8659  8660
  8661  8662  8663  8748  8755  8884  8892  8920  9013  9044  9268  9269
  9284  9285  9287  9301  9315  9393  9397  9398  9420  9421  9426  9427
  9428  9429  9431  9432  9433  9434 

In [21]:
poisoned_total = poisoned_location
poisoned_with_label5 = set(i for i in poisoned_location if int(train_set[i][1]) == 5)
print("poisons total:", len(poisoned_total))
print("poisons labeled 5:", len(poisoned_with_label5))


poisons total: 394
poisons labeled 5: 394


In [22]:
# how many samples are labeled as target (after relabeling) and how many of those are poisoned?
target_indices = [i for i in range(len(train_set)) if int(train_set[i][1]) == TARGET_LABEL]
poison_in_target = [i for i in target_indices if int(train_set[i][2]) == 1]

print("Target-labeled subset size:", len(target_indices))
print("Poison inside target:", len(poison_in_target))
print("Poison fraction inside target:", len(poison_in_target) / max(1, len(target_indices)))


Target-labeled subset size: 2237
Poison inside target: 394
Poison fraction inside target: 0.17612874385337507


# AUTOENCODER

In [3]:
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from GTSRB import GTSRB_Wrapper
from torchvision.models import resnet18

# Import the library directly
sys.path.append(os.path.join(os.getcwd(), '..'))
from BackdoorBox.core.defenses.AutoEncoderDefense import AutoEncoderDefense

# --- CONFIG ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
POISON_RATE = 0.01 
TARGET_LABEL = 5 

# --- 1. DEFINE THE MODEL (Unavoidable) ---
# The library requires you to pass a model object.
# This structure is specifically built for 32x32 images.
class SimpleAutoEncoder(nn.Module):
    def __init__(self):
        super(SimpleAutoEncoder, self).__init__()
        # Encoder (Only downsample twice: 32 -> 16 -> 8)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),  # -> 16x16
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # -> 8x8
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), # -> 16x16
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),  # -> 32x32
            nn.Tanh() # Output [-1, 1]
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# --- 2. DATA SETUP ---
# Standard normalization for ResNet
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)),
])

train_set = GTSRB_Wrapper(mode='train', poison_type='black_1', poison_rate=POISON_RATE, transform=transform, target_label=TARGET_LABEL)
test_clean = GTSRB_Wrapper(mode='test', poison_type='black_1', poison_rate=0.0, transform=transform, target_label=TARGET_LABEL)
test_poison = GTSRB_Wrapper(mode='test', poison_type='black_1', poison_rate=1.0, transform=transform, target_label=TARGET_LABEL)

# --- 3. RUN DEFENSE (Using Library Code) ---
ae_model = SimpleAutoEncoder().to(DEVICE)

# Initialize the library class directly
defense = AutoEncoderDefense(
    autoencoder=ae_model,
    seed=0
)

# Configure the library's training loop
schedule = {
    'device': 'GPU' if torch.cuda.is_available() else 'CPU',
    'GPU_num': 1,
    'CUDA_VISIBLE_DEVICES': '0',
    'batch_size': BATCH_SIZE,
    'num_workers': 4,
    'lr': 0.001,
    'epochs': 20,
    'betas': (0.9, 0.999),
    'eps': 1e-8,
    'weight_decay': 0,
    'amsgrad': False,
    'log_iteration_interval': 100,
    'test_epoch_interval': 5,
    'save_epoch_interval': 10,
    'save_dir': 'ae_logs',
    'experiment_name': 'gtsrb_ae',
    'schedule': [10, 15], 
    'gamma': 0.1
}


In [5]:
print(">>> Training AutoEncoder using BackdoorBox...")
# This uses the library's internal training loop
defense.train_autoencoder(train_set, test_clean, schedule)

# --- 4. EVALUATE ---
# Load your classifier (BadNet)
badnet = resnet18(num_classes=43).to(DEVICE)
# badnet.load_state_dict(torch.load("badnet.pth")) # Load your trained weights

def eval_pipeline(model, defense, loader):
    model.eval()
    defense.autoencoder.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            # The library provides .preprocess() to clean the image
            x_clean = defense.preprocess(x) 
            _, pred = torch.max(model(x_clean), 1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

acc = eval_pipeline(badnet, defense, DataLoader(test_clean, batch_size=256))
asr = eval_pipeline(badnet, defense, DataLoader(test_poison, batch_size=256))

print(f"Clean Acc: {acc:.2f}% | ASR: {asr:.2f}%")

>>> Training AutoEncoder using BackdoorBox...
This machine has 1 cuda devices, and use 1 of them to train.
Total train samples: 39209
Total test samples: 12630
Batch size: 64
iteration every epoch: 612
Initial learning rate: 0.001

[2026-02-04_13:02:01] Epoch:1/20, iteration:100/612, lr: 0.001, loss: 0.003033015411347151, time: 0.7097556591033936

[2026-02-04_13:02:02] Epoch:1/20, iteration:200/612, lr: 0.001, loss: 0.00305755902081728, time: 0.5184855461120605

[2026-02-04_13:02:02] Epoch:1/20, iteration:300/612, lr: 0.001, loss: 0.0031716185621917248, time: 0.5089609622955322

[2026-02-04_13:02:03] Epoch:1/20, iteration:400/612, lr: 0.001, loss: 0.003109533339738846, time: 0.4953188896179199

[2026-02-04_13:02:03] Epoch:1/20, iteration:500/612, lr: 0.001, loss: 0.002441407646983862, time: 0.4984884262084961

[2026-02-04_13:02:04] Epoch:1/20, iteration:600/612, lr: 0.001, loss: 0.002461592899635434, time: 0.4922828674316406

[2026-02-04_13:02:05] Epoch:2/20, iteration:87/612, lr: 0.00

In [ ]:
import matplotlib.pyplot as plt

def visualize_reconstruction(model, loader, device):
    model.eval()
    x, _, _ = next(iter(loader))
    x = x.to(device)
    with torch.no_grad():
        recon = model(x)
    
    # Un-normalize for visualization: [-1, 1] -> [0, 1]
    x = x * 0.5 + 0.5
    recon = recon * 0.5 + 0.5
    
    # Plot first 5 images
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for i in range(5):
        axes[0, i].imshow(x[i].permute(1, 2, 0).cpu().numpy())
        axes[0, i].set_title("Original")
        axes[0, i].axis('off')
        
        axes[1, i].imshow(recon[i].permute(1, 2, 0).cpu().numpy())
        axes[1, i].set_title("Recon")
        axes[1, i].axis('off')
    plt.savefig("ae_debug.png")
    print("Saved visualization to ae_debug.png")

# Call this after defense.train_autoencoder(...)
visualize_reconstruction(ae_model, DataLoader(test_clean, batch_size=16), DEVICE)

ValueError: too many values to unpack (expected 2)